# 4.10 · 02 — Baseline (не-DL)
Baseline-и оцінюються **на тому самому split** (train → fit, validation → вибір гіперпараметра). Тест тут **не використовується**.
- **B-1 Majority class** — нижня межа «без моделі».
- **B0 Logistic regression** на 64 пікселях — проста лінійна модель; `C` підбирається на validation.

In [1]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from src.utils import load_config, set_seed, load_split, RESULTS

cfg = load_config(); set_seed(cfg["seed"])
data = load_split()
(Xtr, ytr), (Xva, yva) = data["train"], data["val"]
flat = lambda X: X.reshape(len(X), -1)
print({k: v[0].shape for k, v in data.items()})

{'train': (1077, 8, 8), 'val': (360, 8, 8), 'test': (360, 8, 8)}


In [2]:
dummy = DummyClassifier(strategy="most_frequent").fit(flat(Xtr), ytr)
print("Majority — val accuracy: {:.3f}".format(accuracy_score(yva, dummy.predict(flat(Xva)))))

rows = []
for C in cfg["baseline"]["C_grid"]:
    m = LogisticRegression(C=C, max_iter=cfg["baseline"]["max_iter"]).fit(flat(Xtr), ytr)
    rows.append({"C": C,
                 "train macro-F1": f1_score(ytr, m.predict(flat(Xtr)), average="macro"),
                 "val macro-F1": f1_score(yva, m.predict(flat(Xva)), average="macro")})
grid = pd.DataFrame(rows).round(4)
grid

Majority — val accuracy: 0.103


,C,train macro-F1,val macro-F1
0,0.01,0.9163,0.9051
1,0.10,0.9552,0.9501
2,1.00,0.9870,0.9725
3,10.00,0.9991,0.9751


In [3]:
best_C = float(grid.loc[grid["val macro-F1"].idxmax(), "C"])
best = grid.loc[grid["val macro-F1"].idxmax()]
print(f"Обрано C = {best_C}: train F1 = {best['train macro-F1']:.4f}, val F1 = {best['val macro-F1']:.4f}, "
      f"gap = {best['train macro-F1'] - best['val macro-F1']:.4f}")

RESULTS.mkdir(exist_ok=True)
with open(RESULTS / "baseline.json", "w") as f:
    json.dump({"model": "LogisticRegression", "C": best_C,
               "train_macro_f1": float(best["train macro-F1"]), "val_macro_f1": float(best["val macro-F1"])}, f, indent=2)
print("Збережено results/baseline.json")

Обрано C = 10.0: train F1 = 0.9991, val F1 = 0.9751, gap = 0.0240
Збережено results/baseline.json


**Висновок етапу:** навіть лінійна модель на сирих пікселях дає високу якість (~0.96–0.97 macro-F1 на validation) — це сильний baseline, і CNN має перевершити саме його, а не «випадкове вгадування» (10%). Фінальна оцінка baseline на тесті (у т.ч. за domain shift) виконується разом з DL-моделлю в `03_dl_experiment.ipynb`.